<a href="https://colab.research.google.com/github/TurkuNLP/intro-to-nlp/blob/master/ex_task2_advanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
!wget -nc https://github.com/TurkuNLP/intro-to-nlp/raw/master/Data/imdb_train.json

File ‘imdb_train.json’ already there; not retrieving.



In [24]:
import json # JSON encoder and decoder: store python data structures (e.g. lists and dictionaries) as strings
from transformers import AutoTokenizer
from collections import Counter

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

with open("imdb_train.json", "rt", encoding="utf-8") as f:
    data = json.load(f)
    data = data #[:1000] # downsample if needed

# tokenize and calculate DF values for each token
tokenized_documents = []
df_values = Counter()
for doc in data:
  tokens = tokenizer.tokenize(doc["text"])
  tokenized_documents.append(tokens)
  unique_tokens = set(tokens)
  df_values.update(unique_tokens) # increase df count by one for each unique token
  # note that if we update Counter with 'tokens', we calculate normal token frequencies,
  # but when we update it with unique tokens (set),. we keep track of document frequencies

# turn df counts into idf scores
idf_scores = {}
for token, df in df_values.items():
  idf_scores[token] = len(data) / df

print(idf_scores["the"])

# sorted
sorted_idf_scores = sorted(idf_scores.items(), key=lambda x: x[1], reverse=True)

print("Lowest:", sorted_idf_scores[-10:])
print("Highest:", sorted_idf_scores[:10])


Token indices sequence length is longer than the specified maximum sequence length for this model (535 > 512). Running this sequence through the model will result in indexing errors


1.0083491308030492
Lowest: [("'", 1.1195199498455062), ('is', 1.1141316457952672), ('this', 1.1042890587040064), ('to', 1.0645092612305727), ('of', 1.0536075522589345), (',', 1.039155374511597), ('and', 1.0345112968633619), ('a', 1.0340833884844474), ('the', 1.0083491308030492), ('.', 1.003129764866383)]
Highest: [('shielding', 25000.0), ('##arns', 25000.0), ('murmurs', 25000.0), ('166', 25000.0), ('walkway', 25000.0), ('eton', 25000.0), ('planck', 25000.0), ('yemen', 25000.0), ('neared', 25000.0), ('wylie', 25000.0)]


In [25]:
import numpy as np

def tf_idf(tokenized_document, log=False):

  tf_values = Counter(tokenized_document)
  tfidf_scores = {}
  for token, tf in tf_values.items():
    idf = idf_scores[token]
    if log:
      tfidf_scores[token] = tf * np.log(idf)
    else:
      tfidf_scores[token] = tf * idf
  sorted_tfidf = sorted(tfidf_scores.items(), key=lambda x: x[1], reverse=True)
  print("Highest tf-idf weights:")
  for token,value in sorted_tfidf[:10]:
    print(f"{token}, tf-idf:{value:.2f}, tf:{tf_values[token]}, idf:{idf_scores[token]:.2f}")
  print()

print("First document:")
print(data[0]["text"][:200])
tf_idf(tokenized_documents[0], log=False)
tf_idf(tokenized_documents[0], log=True)

print("Second document:")
print(data[1]["text"][:200])
tf_idf(tokenized_documents[1], log=False)
tf_idf(tokenized_documents[1], log=True)

First document:
With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want t
Highest tf-idf weights:
overheard, tf-idf:3571.43, tf:1, idf:3571.43
supplying, tf-idf:2500.00, tf:1, idf:2500.00
##j, tf-idf:2235.77, tf:11, idf:203.25
excluding, tf-idf:1785.71, tf:1, idf:1785.71
nah, tf-idf:1136.36, tf:1, idf:1136.36
consent, tf-idf:1041.67, tf:1, idf:1041.67
##walker, tf-idf:833.33, tf:2, idf:416.67
##pathic, tf-idf:806.45, tf:1, idf:806.45
liar, tf-idf:641.03, tf:1, idf:641.03
dunn, tf-idf:531.91, tf:1, idf:531.91

Highest tf-idf weights:
##j, tf-idf:58.46, tf:11, idf:203.25
m, tf-idf:22.33, tf:12, idf:6.43
##walker, tf-idf:12.06, tf:2, idf:416.67
sequence, tf-idf:10.71, tf:3, idf:35.56
guilty, tf-idf:9.85, tf:2, idf:137.36
jackson, tf-idf:9.83, tf:2, idf:136.61
pe, tf-idf:9.36, tf:2, idf:107.76
moon, tf-idf:9.28, tf:2, idf:103.73
##sc, tf-idf:9.14, t

##Discussion

**What kind of tokens receive high/low IDF values?**

* Low IDF: Words that appear in almost every document (so-called stopwords)
  * the, a, and, of, to , this
* High IDF: Words that appear in only one or very few documents
  * E.g. shielding, ##arns, murmurs, 166, walkway, eton


**Write a function which receives a document and prints 10 words from the document with highest TF-IDF weights. Do these seem to tell something about the document content?**

* Without logarithmic scaling, very rare words receive extremely high tf-idf weights. Using logarithmic scaling produces somewhat better results (and is standard in tf-idf implementations).
* Subword tokens can be unintuitive (e.g. `##j` is difficult to interpret). Linguistically motivated tokenization would be more intuitive.
* tf-idf weigting generally produces better results than raw token frequencies (mainly because it downweights stopwords), but it is still a relatively simple weighting scheme.